In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Deep Convolutional Generative Adversarial Networks

In that section, we introduced the basic ideas behind how GANs work. We showed that they can draw samples from some simple, easy-to-sample distribution, like a uniform or normal distribution, and transform them into samples that appear to match the distribution of some dataset. And while our example of matching a 2D Gaussian distribution got the point across, it is not especially exciting.

In this section, we will demonstrate how you can use GANs to generate photorealistic images. We will be basing our models on the deep convolutional GANs (DCGAN) introduced in @Radford.Metz.Chintala.2015 . We will borrow the convolutional architectures that have proven so successful for discriminative computer vision problems and show how via GANs, they can be leveraged to generate photorealistic images.

In [ ]:
%matplotlib inline
from d2l import jax as d2l
import jax
from jax import numpy as jnp
from flax import nnx
import optax
import numpy as np
from PIL import Image
import tensorflow as tf
import os

## The Pokemon Dataset

The dataset we will use is a collection of Pokemon sprites obtained from [pokemondb](https://pokemondb.net/sprites). First download, extract and load this dataset.

In [ ]:

d2l.DATA_HUB['pokemon'] = (d2l.DATA_URL + 'pokemon.zip',
                           'c065c0e2593b8b161a2d7873e42418bf6a21106c')

data_dir = d2l.download_extract('pokemon')

We resize each image into $64\times 64$. The `ToTensor` transformation will project the pixel value into $[0, 1]$, while our generator will use the tanh function to obtain outputs in $[-1, 1]$. Therefore we normalize the data with $0.5$ mean and $0.5$ standard deviation to match the value range.

In [ ]:
batch_size = 256

# Load all Pokemon images via PIL, resize to 64x64, normalise to [-1, 1]
_all_images = []
for root, dirs, files in os.walk(data_dir):
    for fname in sorted(files):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            img = Image.open(os.path.join(root, fname)).convert('RGB')
            img = img.resize((64, 64))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = (arr - 0.5) / 0.5  # normalise to [-1, 1]
            _all_images.append(arr)  # (H, W, C)

_all_images = np.stack(_all_images)  # (N, H, W, C)
_all_labels = np.zeros(len(_all_images), dtype=np.int32)  # dummy labels

data_iter = d2l.load_array((_all_images, _all_labels), batch_size,
                           is_train=True)

Let's visualize the first 20 images.

In [ ]:
d2l.set_figsize((4, 4))
for batch in data_iter:
    X = np.array(batch[0])  # (N, H, W, C), values in [-1, 1]
    imgs = X[:20] / 2 + 0.5
    d2l.show_images(imgs, num_rows=4, num_cols=5)
    break

## The Generator

The generator needs to map the noise variable $\mathbf z\in\mathbb R^d$, a length-$d$ vector, to an RGB image with width and height of $64\times 64$ . In that section we introduced the fully convolutional network that uses transposed convolution layer (refer to that section) to enlarge input size. The basic block of the generator contains a transposed convolution layer followed by the batch normalization and ReLU activation.

In [ ]:
class G_block(nnx.Module):
    def __init__(self, out_channels, in_channels=3, kernel_size=4,
                 strides=2, padding='SAME', rngs=None):
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        init = nnx.initializers.normal(0.02)
        self.conv2d_trans = nnx.ConvTranspose(
            in_channels, out_channels, kernel_size=(kernel_size, kernel_size),
            strides=(strides, strides), padding=padding, use_bias=False,
            kernel_init=init, rngs=rngs)
        self.batch_norm = nnx.BatchNorm(out_channels, rngs=rngs)

    def __call__(self, X):
        return nnx.relu(self.batch_norm(self.conv2d_trans(X)))

In default, the transposed convolution layer uses a $k_h = k_w = 4$ kernel, a $s_h = s_w = 2$ strides, and a $p_h = p_w = 1$ padding. With an input shape of $n_h^{'} \times n_w^{'} = 16 \times 16$, the generator block will double input's width and height.

$$
\begin{aligned}
n_h^{'} \times n_w^{'} &= [(n_h k_h - (n_h-1)(k_h-s_h)- 2p_h] \times [(n_w k_w - (n_w-1)(k_w-s_w)- 2p_w]\\
  &= [(k_h + s_h (n_h-1)- 2p_h] \times [(k_w + s_w (n_w-1)- 2p_w]\\
  &= [(4 + 2 \times (16-1)- 2 \times 1] \times [(4 + 2 \times (16-1)- 2 \times 1]\\
  &= 32 \times 32 .\\
\end{aligned}
$$

In [ ]:
x = jnp.zeros((2, 16, 16, 3))  # Channel last convention
g_blk = G_block(out_channels=20, rngs=nnx.Rngs(0))
g_blk(x).shape

If we change the transposed convolution layer to a $4\times 4$ kernel, $1\times 1$ strides and zero padding, then with an input size of $1 \times 1$, the output will have its width and height increased by 3 respectively.

In [ ]:
x = jnp.zeros((2, 1, 1, 3))
# `padding="VALID"` corresponds to no padding
g_blk = G_block(out_channels=20, strides=1, padding='VALID',
                rngs=nnx.Rngs(0))
g_blk(x).shape

The generator consists of four basic blocks that increase input's both width and height from 1 to 32. At the same time, it first projects the latent variable into $64\times 8$ channels, and then halve the channels each time. At last, a transposed convolution layer is used to generate the output. It further doubles the width and height to match the desired $64\times 64$ shape, and reduces the channel size to $3$. The tanh activation function is applied to project output values into the $(-1, 1)$ range.

In [ ]:
n_G = 64

class Generator(nnx.Module):
    def __init__(self, latent_dim=100, n_G=64, rngs=None):
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.blocks = nnx.List([
            G_block(n_G * 8, latent_dim, strides=1, padding='VALID', rngs=rngs),
            G_block(n_G * 4, n_G * 8, rngs=rngs),
            G_block(n_G * 2, n_G * 4, rngs=rngs),
            G_block(n_G, n_G * 2, rngs=rngs)])
        self.output = nnx.ConvTranspose(
            n_G, 3, kernel_size=(4, 4), strides=(2, 2), padding='SAME',
            use_bias=False, kernel_init=nnx.initializers.normal(0.02),
            rngs=rngs)

    def __call__(self, X):
        X = self.blocks[0](X)
        # Output: (4, 4, 64 * 8)
        X = self.blocks[1](X)
        # Output: (8, 8, 64 * 4)
        X = self.blocks[2](X)
        # Output: (16, 16, 64 * 2)
        X = self.blocks[3](X)
        # Output: (32, 32, 64)
        X = nnx.tanh(self.output(X))
        # Output: (64, 64, 3)
        return X

net_G = Generator(n_G=n_G, rngs=nnx.Rngs(0))

Generate a 100 dimensional latent variable to verify the generator's output shape.

In [ ]:
x = jnp.zeros((1, 1, 1, 100))
net_G(x).shape

## Discriminator

The discriminator is a normal convolutional network except that it uses a leaky ReLU as its activation function. Given $\alpha \in[0, 1]$, its definition is

$$\textrm{leaky ReLU}(x) = \begin{cases}x & \textrm{if}\ x > 0\\ \alpha x &\textrm{otherwise}\end{cases}.$$

As it can be seen, it is normal ReLU if $\alpha=0$, and an identity function if $\alpha=1$. For $\alpha \in (0, 1)$, leaky ReLU is a nonlinear function that give a non-zero output for a negative input. It aims to fix the "dying ReLU" problem that a neuron might always output a negative value and therefore cannot make any progress since the gradient of ReLU is 0.

In [ ]:
alphas = [0, .2, .4, .6, .8, 1]
x = jnp.arange(-2, 1, 0.1)
Y = [np.array(nnx.leaky_relu(x, negative_slope=alpha)) for alpha in alphas]
d2l.plot(np.array(x), Y, 'x', 'y', alphas)

The basic block of the discriminator is a convolution layer followed by a batch normalization layer and a leaky ReLU activation. The hyperparameters of the convolution layer are similar to the transpose convolution layer in the generator block.

In [ ]:
class D_block(nnx.Module):
    def __init__(self, out_channels, in_channels=3, kernel_size=4,
                 strides=2, padding='SAME', alpha=0.2, rngs=None):
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.alpha = alpha
        self.conv2d = nnx.Conv(
            in_channels, out_channels, kernel_size=(kernel_size, kernel_size),
            strides=(strides, strides), padding=padding, use_bias=False,
            kernel_init=nnx.initializers.normal(0.02), rngs=rngs)
        self.batch_norm = nnx.BatchNorm(out_channels, rngs=rngs)

    def __call__(self, X):
        X = self.batch_norm(self.conv2d(X))
        return nnx.leaky_relu(X, negative_slope=self.alpha)

A basic block with default settings will halve the width and height of the inputs, as we demonstrated in that section. For example, given a input shape $n_h = n_w = 16$, with a kernel shape $k_h = k_w = 4$, a stride shape $s_h = s_w = 2$, and a padding shape $p_h = p_w = 1$, the output shape will be:

$$
\begin{aligned}
n_h^{'} \times n_w^{'} &= \lfloor(n_h-k_h+2p_h+s_h)/s_h\rfloor \times \lfloor(n_w-k_w+2p_w+s_w)/s_w\rfloor\\
  &= \lfloor(16-4+2\times 1+2)/2\rfloor \times \lfloor(16-4+2\times 1+2)/2\rfloor\\
  &= 8 \times 8 .\\
\end{aligned}
$$

In [ ]:
x = jnp.zeros((2, 16, 16, 3))
d_blk = D_block(out_channels=20, rngs=nnx.Rngs(0))
d_blk(x).shape

The discriminator is a mirror of the generator.

In [ ]:
n_D = 64

class Discriminator(nnx.Module):
    def __init__(self, n_D=64, rngs=None):
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.blocks = nnx.List([
            D_block(n_D, 3, rngs=rngs),
            D_block(n_D * 2, n_D, rngs=rngs),
            D_block(n_D * 4, n_D * 2, rngs=rngs),
            D_block(n_D * 8, n_D * 4, rngs=rngs)])
        self.output = nnx.Conv(
            n_D * 8, 1, kernel_size=(4, 4), padding='VALID', use_bias=False,
            kernel_init=nnx.initializers.normal(0.02), rngs=rngs)

    def __call__(self, X):
        X = self.blocks[0](X)
        # Output: (32, 32, 64)
        X = self.blocks[1](X)
        # Output: (16, 16, 64 * 2)
        X = self.blocks[2](X)
        # Output: (8, 8, 64 * 4)
        X = self.blocks[3](X)
        # Output: (4, 4, 64 * 8)
        X = self.output(X)
        # Output: (1, 1, 1)
        return X

net_D = Discriminator(n_D=n_D, rngs=nnx.Rngs(1))

It uses a convolution layer with output channel $1$ as the last layer to obtain a single prediction value.

In [ ]:
x = jnp.zeros((1, 64, 64, 3))
net_D(x).shape

## Training

Compared to the basic GAN in that section, we use the same learning rate for both generator and discriminator since they are similar to each other. In addition, we change $\beta_1$ in Adam (that section) from $0.9$ to $0.5$. It decreases the smoothness of the momentum, the exponentially weighted moving average of past gradients, to take care of the rapid changing gradients because the generator and the discriminator fight with each other. Besides, the random generated noise `Z`, is a 4-D tensor and we are using GPU to accelerate the computation.

In [ ]:
@nnx.jit
def update_D(X, Z, net_D, net_G, optimizer_D):
    """Update the discriminator while keeping generator weights fixed."""
    batch_size = X.shape[0]
    fake_X = net_G(Z)

    def loss_D_fn(model_D):
        real_Y = model_D(X).squeeze()
        fake_Y = model_D(fake_X).squeeze()
        return (jnp.sum(optax.sigmoid_binary_cross_entropy(
                    real_Y, jnp.ones((batch_size,)))) +
                jnp.sum(optax.sigmoid_binary_cross_entropy(
                    fake_Y, jnp.zeros((batch_size,))))) / 2

    loss_D, grads_D = nnx.value_and_grad(loss_D_fn)(net_D)
    optimizer_D.update(net_D, grads_D)
    return loss_D

@nnx.jit
def update_G(Z, net_D, net_G, optimizer_G):
    """Update the generator through the fixed discriminator."""
    def loss_G_fn(model_G, model_D):
        # Passing the discriminator explicitly keeps its parameters outside
        # the differentiated argument while preserving training-mode
        # BatchNorm, as in the discriminator update.
        fake_Y = model_D(model_G(Z)).squeeze()
        return jnp.sum(optax.sigmoid_binary_cross_entropy(
            fake_Y, jnp.ones((Z.shape[0],))))

    loss_G, grads_G = nnx.value_and_grad(
        loss_G_fn, argnums=0)(net_G, net_D)
    optimizer_G.update(net_G, grads_G)
    return loss_G

def train(net_D, net_G, data_iter, num_epochs, lr, latent_dim):
    key = jax.random.PRNGKey(0)
    optimizer_D = nnx.Optimizer(
        net_D, optax.adam(lr, b1=0.5, b2=0.999), wrt=nnx.Param)
    optimizer_G = nnx.Optimizer(
        net_G, optax.adam(lr, b1=0.5, b2=0.999), wrt=nnx.Param)

    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[1, num_epochs], nrows=2, figsize=(5, 5),
                            legend=['discriminator', 'generator'])
    animator.fig.subplots_adjust(hspace=0.3)

    for epoch in range(1, num_epochs + 1):
        timer = d2l.Timer()
        losses_D, losses_G = [], []
        num_examples = 0
        for batch in data_iter:
            X = jnp.array(batch[0])  # Already (N, H, W, C)
            batch_size = X.shape[0]
            key, subkey = jax.random.split(key)
            Z = jax.random.normal(subkey, (batch_size, 1, 1, latent_dim))

            loss_D_val = update_D(X, Z, net_D, net_G, optimizer_D)
            loss_G_val = update_G(Z, net_D, net_G, optimizer_G)

            losses_D.append(loss_D_val)
            losses_G.append(loss_G_val)
            num_examples += batch_size

        # Show generated examples
        key, subkey = jax.random.split(key)
        Z = jax.random.normal(subkey, (21, 1, 1, latent_dim))
        fake_x = nnx.view(net_G, use_running_average=True)(Z)
        fake_x = fake_x / 2 + 0.5
        imgs = jnp.concatenate(
            [jnp.concatenate([fake_x[i * 7 + j] for j in range(7)], axis=1)
             for i in range(len(fake_x) // 7)], axis=0)
        animator.axes[1].cla()
        animator.axes[1].imshow(np.array(imgs))
        # Show the losses
        # Aggregate the scalar losses once per epoch. Converting the sample
        # grid above has already synchronized the generated images.
        loss_D = float(jnp.stack(losses_D).sum()) / num_examples
        loss_G = float(jnp.stack(losses_G).sum()) / num_examples
        animator.add(epoch, (loss_D, loss_G))
    print(f'loss_D {loss_D:.3f}, loss_G {loss_G:.3f}, '
          f'{num_examples / timer.stop():.1f} examples/sec')

We train the model with a small number of epochs just for demonstration.
For better performance,
the variable `num_epochs` can be set to a larger number.
GAN losses should not be read like supervised validation loss: a large
generator loss can coexist with useful samples when the discriminator is
confident. Inspect the generated image grid in the lower panel as the primary
teaching signal, and use longer runs when comparing architectures.

In [ ]:
latent_dim, lr, num_epochs = 100, 0.0002, 40
train(net_D, net_G, data_iter, num_epochs, lr, latent_dim)

## Summary

* DCGAN architecture has four convolutional layers for the Discriminator and four "fractionally-strided" convolutional layers for the Generator.
* The Discriminator is a 4-layer strided convolutions with batch normalization (except its input layer) and leaky ReLU activations.
* Leaky ReLU is a nonlinear function that give a non-zero output for a negative input. It aims to fix the “dying ReLU” problem and helps the gradients flow easier through the architecture.


## Exercises

1. What will happen if we use standard ReLU activation rather than leaky ReLU?
1. Apply DCGAN on Fashion-MNIST and see which category works well and which does not.

[Discussions](https://d2l.discourse.group/t/1083)